# Teacher's Notebook — Prosthetics Data Engineering Workshop

---
## Step 0 — Setup


In [1]:
from pathlib import Path
import re
import hashlib
import pandas as pd

In [2]:
# Find the CSV regardless of which folder this notebook is opened from
csv_candidates = [
    Path('prosthetics_data.csv'),
    Path('..') / 'prosthetics_data.csv',
]
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Could not find prosthetics_data.csv in this folder or the parent.')

df = pd.read_csv(csv_path)
df.head()

,Patient_ID,Full_Name,Age_Input,Implant_Date,Amputation_Cause,Device_Model_ID,Amputation_Pos,Clinical_Notes
0,PT-001,John Doe,28,12/05/2024,Trauma,M302,AK,Patient was referred from the northern field c...
1,PT-002,Amara Okafor,34,2024-06-15,diabetes,M404,BK,Patient arrived with a family member and reque...
2,PT-003,Miguel Santos,1995,May 12th 2024,TRAUMA,M302,TT,The patient reported good progress after prost...
3,PT-004,Priya Nair,"Don't know, maybe 40",07/11/2023,blast injury,M505,T-T,This patient was seen during a mobile outreach...
4,PT-005,Fatima Hassan,1988,2023/09/01,Trauma,M404,B-K,The fitting session was completed successfully...


Three calls before any cleaning:
- `df.info()` — column names, dtypes, non-null counts. Messy types show up immediately (object where you'd expect int or datetime).
- `df.isna().sum()` — two columns have NaN: `Full_Name` (PT-007) and `Age_Input` (PT-010).
- `df.head()` — enough rows to see the format variety without overwhelming the screen.

**Flags to point out:**
- **`Age_Input`** is dtype `object` despite mostly looking numeric — red flag.
- **`Implant_Date`** also `object` — free-text dates, six or seven formats incoming.
- **`Amputation_Pos`** column (position codes: AK, BK, TT, etc.)

In [3]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Patient_ID        15 non-null     str  
 1   Full_Name         14 non-null     str  
 2   Age_Input         14 non-null     str  
 3   Implant_Date      15 non-null     str  
 4   Amputation_Cause  15 non-null     str  
 5   Device_Model_ID   15 non-null     str  
 6   Amputation_Pos    15 non-null     str  
 7   Clinical_Notes    15 non-null     str  
dtypes: str(8)
memory usage: 1.1 KB

Missing values per column:
Patient_ID          0
Full_Name           1
Age_Input           1
Implant_Date        0
Amputation_Cause    0
Device_Model_ID     0
Amputation_Pos      0
Clinical_Notes      0
dtype: int64


---
## Section 1 — Ingestion & Pipelines (ETL vs ELT)

Frame why pipeline design matters in healthcare before touching any data.

### ETL (Extract → Transform → Load)
- Validate and mask PII *before* it reaches the destination.
- Right choice when destination is a shared cloud system and raw records must never land there.
- Trade-off: you need to know your transformations upfront.

### ELT (Extract → Load → Transform)
- Raw data lands first; transform inside the warehouse later.
- Right choice when data is large and storage is cheap.
- Risk in healthcare: raw patient records sit in the warehouse until the transform runs.

> **Ask the room:** should this charity ETL or ELT when sending records to a cloud analytics platform?
> Listening for: **ETL** — names and PII must be masked *before* hitting the cloud. That’s the path this workshop follows.

---
## Section 2 — Data Cleaning

Three cleaning problems, one function each (keeps each piece independently testable):
1. `Age_Input` — integers, birth years, and freeform text all in one column
2. `Implant_Date` — six or seven different date formats including named months with ordinal suffixes
3. `Amputation_Cause` — inconsistent casing and trailing spaces

### 2a — Parse `Age_Input`

Show the raw column first.

In [4]:
# What does the column actually contain?
df[['Patient_ID', 'Age_Input']]

,Patient_ID,Age_Input
0,PT-001,28
1,PT-002,34
2,PT-003,1995
3,PT-004,"Don't know, maybe 40"
4,PT-005,1988
5,PT-006,around 50
6,PT-007,19
7,PT-008,41
8,PT-009,52
9,PT-010,NaN


Four kinds of value in the column:
- **Plain age:** `28`, `34`, `19`
- **Birth year:** `1995`, `1988`, `2001` — needs converting to age
- **Free text with a number:** `'around 50'`, `'maybe 27'`, `"Don't know, maybe 40"`
- **`NaN`** — PT-010 is genuinely missing

**Logic:** NaN → `None`; number > 1900 → birth year, subtract from 2026; ≤ 1900 → already an age; otherwise pull the first digit run from the text.

In [ ]:
CURRENT_YEAR = 2026

def parse_age_input(value):
    """Convert the messy Age_Input column to a clean integer age."""
    if pd.isna(value):
        return None

    # Numeric value — already an int/float, or a string like '34'
    if isinstance(value, (int, float)) and not pd.isna(value):
        numeric_value = int(value)
    else:
        match = re.search(r'\d+', str(value))
        if not match:
            return None
        numeric_value = int(match.group())

    # Classify: age (<100), birth year (>1900), or edge case
    if numeric_value < 100:
        return numeric_value           # already an age
    if numeric_value > 1900:
        return CURRENT_YEAR - numeric_value   # birth year
    return numeric_value               # edge case: 100–1900, return as-is

def amputation_age(age, amputation_date):
    
    birth_year = CURRENT_YEAR - age if age is not None else None
    if birth_year is None or pd.isna(amputation_date):
        return None
    return amputation_date.year - birth_year

Test representative inputs before applying to the whole column so hopefully catches bugs early and makes the logic visible to the room.

In [6]:
# Quick sanity checks before applying to the whole column
test_cases = [
    (28,                    28),   # plain integer age
    ('34',                  34),   # age as string
    (1995,                  31),   # birth year
    ("Don't know, maybe 40", 40),  # free text
    ('around 50',           50),
    (float('nan'),         None),  # missing
]
for value, expected in test_cases:
    result = parse_age_input(value)
    status = 'OK' if result == expected else f'FAIL (expected {expected})'
    print(f'{str(value):30} -> {str(result):6}  {status}')

28                             -> 28      OK
34                             -> 34      OK
1995                           -> 31      OK
Don't know, maybe 40           -> 40      OK
around 50                      -> 50      OK
nan                            -> None    OK


In [7]:
df['Age_Years'] = df['Age_Input'].apply(parse_age_input)
df[['Patient_ID', 'Age_Input', 'Age_Years']]

,Patient_ID,Age_Input,Age_Years
0,PT-001,28,28.0
1,PT-002,34,34.0
2,PT-003,1995,31.0
3,PT-004,"Don't know, maybe 40",40.0
4,PT-005,1988,38.0
5,PT-006,around 50,50.0
6,PT-007,19,19.0
7,PT-008,41,41.0
8,PT-009,52,52.0
9,PT-010,NaN,NaN


### 2b — Parse `Implant_Date`

Show the raw column and count the formats.

In [8]:
# How many different formats do we actually have?
df['Implant_Date'].unique()

<StringArray>
[    '12/05/2024',     '2024-06-15',  'May 12th 2024',     '07/11/2023',
     '2023/09/01',   'March 3 2024',     '2024-01-20',     '20/02/2024',
 'April 5th 2024',     '2024-08-30',     '11/12/2023',     '2023-10-14',
  'June 9th 2024',     '2024/04/18',     '15/07/2024']
Length: 15, dtype: str

**Tempting shortcut:** `pd.to_datetime(dayfirst=True)` over everything.

**The bug:** `dayfirst=True` with `format='mixed'` also reorders month/day *inside year-first strings* — `2023-09-01` becomes January 9th, not September 1st.

Demo this live.

In [9]:
# DEMONSTRATION OF THE PITFALL — do not use this in the actual pipeline
demo = pd.Series(['2023-09-01', '2023/09/01'])
print('With dayfirst=True (WRONG for year-first dates):')
print(pd.to_datetime(demo, format='mixed', dayfirst=True).tolist())
print()
print('With dayfirst=False (correct for year-first, wrong for DD/MM/YYYY):')
print(pd.to_datetime(demo, format='mixed', dayfirst=False).tolist())

With dayfirst=True (WRONG for year-first dates):
[Timestamp('2023-01-09 00:00:00'), Timestamp('2023-01-09 00:00:00')]

With dayfirst=False (correct for year-first, wrong for DD/MM/YYYY):
[Timestamp('2023-09-01 00:00:00'), Timestamp('2023-09-01 00:00:00')]


No single `pd.to_datetime` call handles both `DD/MM/YYYY` and `YYYY/MM/DD` safely. The fix: **detect which category each value is** before handing it to pandas.

**Step 1 — Strip ordinal suffixes**
`'5th'` → `'5'` so `'April 5th 2024'` can parse normally.

**Step 2 — Regex-detect year-first format**
Pattern: `^\d{4}[-/]` — if matched, reorder to ISO directly. Pandas never gets the chance to misinterpret it.

In [10]:
# Step 1: strip ordinal suffixes
# 'May 12th 2024' -> 'May 12 2024', 'April 5th 2024' -> 'April 5 2024'
demo_dates = pd.Series(['May 12th 2024', 'April 5th 2024', '2024-06-15'])
demo_dates.str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)

0     May 12 2024
1    April 5 2024
2      2024-06-15
dtype: str

In [11]:
# Step 2: the regex that detects year-first formats
# Matches: 2023-09-01, 2024/04/18, 2024-06-15 — anything starting with 4 digits
pattern = r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$'
for v in ['2023-09-01', '2023/09/01', '2024/04/18', '12/05/2024', 'May 12 2024']:
    matched = bool(re.match(pattern, v.strip()))
    print(f'{v:20} year-first? {matched}')

2023-09-01           year-first? True
2023/09/01           year-first? True
2024/04/18           year-first? True
12/05/2024           year-first? False
May 12 2024          year-first? False


In [12]:
def parse_implant_date(raw):
    """Return YYYY-MM-DD string for any date format found in this dataset."""
    if pd.isna(raw):
        return None

    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(raw).strip())

    # Year-first (2023-09-01, 2024/04/18) — detect first to avoid dayfirst swap
    m = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if m:
        y, mo, d = m.groups()
        return f'{y}-{int(mo):02d}-{int(d):02d}'

    # Named months and DD/MM/YYYY — pandas handles these safely with dayfirst=True
    try:
        return pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return None

In [13]:
# Test against every format in the dataset
date_tests = [
    ('12/05/2024',   '2024-05-12'),
    ('2024-06-15',   '2024-06-15'),
    ('May 12th 2024','2024-05-12'),
    ('07/11/2023',   '2023-11-07'),
    ('2023/09/01',   '2023-09-01'),   # the tricky one
    ('March 3 2024', '2024-03-03'),
    ('20/02/2024',   '2024-02-20'),
    ('April 5th 2024','2024-04-05'),
    ('2024/04/18',   '2024-04-18'),
    ('15/07/2024',   '2024-07-15'),
]
for raw, expected in date_tests:
    got = parse_implant_date(raw)
    status = 'OK' if got == expected else f'FAIL (expected {expected})'
    print(f'{raw:22} -> {got}  {status}')

12/05/2024             -> 2024-05-12  OK
2024-06-15             -> 2024-06-15  OK
May 12th 2024          -> 2024-05-12  OK
07/11/2023             -> 2023-11-07  OK
2023/09/01             -> 2023-09-01  OK
March 3 2024           -> 2024-03-03  OK
20/02/2024             -> 2024-02-20  OK
April 5th 2024         -> 2024-04-05  OK
2024/04/18             -> 2024-04-18  OK
15/07/2024             -> 2024-07-15  OK


In [14]:
df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_implant_date)
df[['Patient_ID', 'Implant_Date', 'Implant_Date_Clean']]

,Patient_ID,Implant_Date,Implant_Date_Clean
0,PT-001,12/05/2024,2024-05-12
1,PT-002,2024-06-15,2024-06-15
2,PT-003,May 12th 2024,2024-05-12
3,PT-004,07/11/2023,2023-11-07
4,PT-005,2023/09/01,2023-09-01
5,PT-006,March 3 2024,2024-03-03
6,PT-007,2024-01-20,2024-01-20
7,PT-008,20/02/2024,2024-02-20
8,PT-009,April 5th 2024,2024-04-05
9,PT-010,2024-08-30,2024-08-30


### 2c — Standardise `Amputation_Cause`

Values are correct — just inconsistent casing and whitespace. A single chained call handles it.

- `str.strip()`: removes leading/trailing whitespace
- `str.lower()`: normalises `'TRAUMA'`, `'Trauma'`, `' TRAUMA'` all to `'trauma'`

PT-012 has `'diabets'` — a genuine typo. Worth pointing it out: lowercase/strip doesn't catch spelling errors. That needs a controlled vocabulary or fuzzy matching step, which is out of scope today.

In [15]:
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].str.lower().str.strip()
df[['Patient_ID', 'Amputation_Cause', 'Amputation_Cause_Clean']]

,Patient_ID,Amputation_Cause,Amputation_Cause_Clean
0,PT-001,Trauma,trauma
1,PT-002,diabetes,diabetes
2,PT-003,TRAUMA,trauma
3,PT-004,blast injury,blast injury
4,PT-005,Trauma,trauma
5,PT-006,diabetes,diabetes
6,PT-007,TRAUMA,trauma
7,PT-008,blast Injury,blast injury
8,PT-009,trauma,trauma
9,PT-010,DIABETES,diabetes


Run `value_counts()` — `'diabets'` appears alongside `'diabetes'`. Shows the limit of a pure formatting fix.

In [16]:
df['Amputation_Cause_Clean'].value_counts()

Amputation_Cause_Clean
trauma          8
diabetes        3
blast injury    3
diabets         1
Name: count, dtype: int64

---
## Section 3 — Cybersecurity & Cloud Privacy (PII Masking)

The data is clean but still contains patient names. Before it moves anywhere outside the secure environment, names must go.

**Option 1 — Delete the column:** simple, irreversible, no re-linkage ever.

**Option 2 — Pseudonymise:** replace each name with a stable token derived from it. Two datasets can still match on the same person without re-exposing the name.

**SHA-256** — a cryptographic hash function. Same input → same 64-character hex string, every time. One-way: you can’t reverse it to recover the original name.

In [17]:
# DEMO: what SHA-256 looks like
import hashlib

sample = 'John Doe'
digest = hashlib.sha256(sample.strip().lower().encode('utf-8')).hexdigest()
print(f'Input : {sample!r}')
print(f'Hash  : {digest}')
print(f'Length: {len(digest)} characters')

Input : 'John Doe'
Hash  : 94890005f3b2117a353da7260259531878cae4f541bf59998511887d1f0221a5
Length: 64 characters


Worth asking why `.strip().lower()` before hashing. `'John Doe'` and `' john doe '` produce different hashes so the same person with slightly different casing across two datasets won't match. Normalise first to ensure they do.

Then: why `.encode('utf-8')`? SHA-256 operates on bytes, not strings.

In [18]:
def anonymize_name(value):
    """Replace a patient name with its SHA-256 hash (normalised first)."""
    normalized = str(value).strip().lower()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

In [19]:
# Apply before the join — this is the ETL 'Transform' step protecting PII
df['Full_Name'] = df['Full_Name'].apply(anonymize_name)
df[['Patient_ID', 'Full_Name']].head()

,Patient_ID,Full_Name
0,PT-001,94890005f3b2117a353da7260259531878cae4f541bf59...
1,PT-002,a083bb1e6e0e335305d8c73f05df267a9e73d7a0ff6bbe...
2,PT-003,727a69291f7d286125c3fa541dc8f063a0264f9413a9ff...
3,PT-004,02e82576b0769a2397331db8f749d6815cfec92e91b0ae...
4,PT-005,8ec077386d04ed3d0182401b5465002bbcf07379999552...


PT-007 had `NaN` in `Full_Name`. After `str(value)` it becomes the string `'nan'`, which hashes consistently. A production system might return a sentinel (`None`) instead — fine to mention as an edge case, but don't let it derail the session. The important concept is the hashing.

---
## Section 4 — Relational Storage & Joins

Clean, privacy-safe records → join to the device inventory to enrich them.

Classic relational structure:
- `Device_Model_ID` is the **foreign key** in the patient table
- `Device_Model_ID` is the **primary key** in the inventory table
- **Inner join:** keep only rows that match on both sides

> **Ask the room:** why inner join over left join?
> Listening for: inner join drops patients with an unknown device ID — surfaces data entry errors rather than silently producing NaN columns.

In [20]:
# Build the reference table
device_inventory = {
    'M210': {'Model_Name': 'AeroStep Lite',   'Cost_USD': 950,  'Manufacturer': 'Stride MedTech'},
    'M302': {'Model_Name': 'FlexFoot Alpha',  'Cost_USD': 1200, 'Manufacturer': 'Global Limb Works'},
    'M404': {'Model_Name': 'CrestFit Pro',    'Cost_USD': 1850, 'Manufacturer': 'OrthoSphere'},
    'M505': {'Model_Name': 'AdaptMax Field',  'Cost_USD': 2100, 'Manufacturer': 'Humanity Mobility Systems'},
}

device_inventory_df = pd.DataFrame.from_dict(device_inventory, orient='index').reset_index()
device_inventory_df = device_inventory_df.rename(columns={'index': 'Device_Model_ID'})
device_inventory_df = device_inventory_df[['Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]
device_inventory_df

,Device_Model_ID,Model_Name,Cost_USD,Manufacturer
0,M210,AeroStep Lite,950,Stride MedTech
1,M302,FlexFoot Alpha,1200,Global Limb Works
2,M404,CrestFit Pro,1850,OrthoSphere
3,M505,AdaptMax Field,2100,Humanity Mobility Systems


A Python dict can't go directly into `pd.merge`. `DataFrame.from_dict(..., orient='index')` converts it — each dict key becomes a row. `reset_index()` makes the key a regular column, then rename it to `Device_Model_ID` so the join has a matching column name on both sides.

In [21]:
patient_device_df = pd.merge(df, device_inventory_df, on='Device_Model_ID', how='inner')
patient_device_df[['Patient_ID', 'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer']]

,Patient_ID,Device_Model_ID,Model_Name,Cost_USD,Manufacturer
0,PT-001,M302,FlexFoot Alpha,1200,Global Limb Works
1,PT-002,M404,CrestFit Pro,1850,OrthoSphere
2,PT-003,M302,FlexFoot Alpha,1200,Global Limb Works
3,PT-004,M505,AdaptMax Field,2100,Humanity Mobility Systems
4,PT-005,M404,CrestFit Pro,1850,OrthoSphere
5,PT-006,M210,AeroStep Lite,950,Stride MedTech
6,PT-007,M302,FlexFoot Alpha,1200,Global Limb Works
7,PT-008,M505,AdaptMax Field,2100,Humanity Mobility Systems
8,PT-009,M404,CrestFit Pro,1850,OrthoSphere
9,PT-010,M210,AeroStep Lite,950,Stride MedTech


In [22]:
# Confirm no rows were dropped (all 15 patients have a valid device ID)
print(f'Rows before merge: {len(df)}')
print(f'Rows after merge : {len(patient_device_df)}')
assert len(patient_device_df) == 15, 'Row count mismatch — check for unknown Device_Model_IDs'

Rows before merge: 15
Rows after merge : 15


---
## Wrap-Up

Show the final DataFrame: names replaced with hashes, dates standardised, ages numeric, device details joined in. A complete ETL pipeline in ∲50 lines.

**Debrief questions:**

1. **Anonymisation vs pseudonymisation** — anonymisation: original can’t be recovered. Pseudonymisation: *can* be recovered with the right key. SHA-256 without a salt leans toward anonymisation, but an attacker who knows the likely names can pre-hash them and compare.

2. **Which step was ETL vs ELT?** — all ETL: we transformed (cleaned and masked) before loading.

3. **What cleaning problem did we not solve?** — the `'diabets'` typo.

4. **What breaks if someone adds device `M606`?** — inner join silently drops that patient. Left join surfaces it as NaN instead. Neither is wrong — depends on your quality gate.

In [23]:
# Final enriched, privacy-safe dataset
output_cols = [
    'Patient_ID', 'Full_Name', 'Age_Years',
    'Implant_Date_Clean', 'Amputation_Cause_Clean',
    'Device_Model_ID', 'Model_Name', 'Cost_USD', 'Manufacturer',
]

with pd.option_context('display.max_colwidth', 18):
    display(patient_device_df[output_cols])

,Patient_ID,Full_Name,Age_Years,Implant_Date_Clean,Amputation_Cause_Clean,Device_Model_ID,Model_Name,Cost_USD,Manufacturer
0,PT-001,94890005f3b211...,28.0,2024-05-12,trauma,M302,FlexFoot Alpha,1200,Global Limb Works
1,PT-002,a083bb1e6e0e33...,34.0,2024-06-15,diabetes,M404,CrestFit Pro,1850,OrthoSphere
2,PT-003,727a69291f7d28...,31.0,2024-05-12,trauma,M302,FlexFoot Alpha,1200,Global Limb Works
3,PT-004,02e82576b0769a...,40.0,2023-11-07,blast injury,M505,AdaptMax Field,2100,Humanity Mobil...
4,PT-005,8ec077386d04ed...,38.0,2023-09-01,trauma,M404,CrestFit Pro,1850,OrthoSphere
5,PT-006,fb8bc0d367f5b3...,50.0,2024-03-03,diabetes,M210,AeroStep Lite,950,Stride MedTech
6,PT-007,9b2d5b4678781e...,19.0,2024-01-20,trauma,M302,FlexFoot Alpha,1200,Global Limb Works
7,PT-008,e438554de6cf47...,41.0,2024-02-20,blast injury,M505,AdaptMax Field,2100,Humanity Mobil...
8,PT-009,ad617c24c70ec4...,52.0,2024-04-05,trauma,M404,CrestFit Pro,1850,OrthoSphere
9,PT-010,ad4a5afe7f5090...,NaN,2024-08-30,diabetes,M210,AeroStep Lite,950,Stride MedTech
